<a href="https://colab.research.google.com/github/njw257/bigdata/blob/main/week04_5685599_%EB%82%A8%EC%A2%85%EC%9A%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 과제
빅데이터프로그래밍 · 경영정보학과

**이름:** 남종욱
**학번:** 5685599

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [4]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


torch 2.11.0+cu130 / device: cuda


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [5]:
# 답안
# 1. 정수 자료형과 1차원 구조를 확인하기 위한 Tensor 생성
a = torch.tensor([1, 2, 3])

# 2. 실수 자료형을 확인하기 위한 Tensor 생성
b = torch.tensor([1.0, 2.0, 3.0])

# 3. 모든 원소가 0인 2차원 Tensor 생성
c = torch.zeros(2, 3)

# 4. 3차원 구조를 확인하기 위한 난수 Tensor 생성
d = torch.randn(2, 3, 4)

# 5. 차원이 없는 스칼라 Tensor 생성
e = torch.tensor(7.0)

# Tensor 5개를 딕셔너리로 정리
tensors = {
    "a 정수": a,
    "b 실수": b,
    "c 영행렬": c,
    "d 난수": d,
    "e 스칼라": e
}

# Tensor별 속성 확인표 생성
rows = []

for name, t in tensors.items():
    rows.append({
        "이름": name,
        "shape": tuple(t.shape),
        "dim": t.dim(),
        "dtype": str(t.dtype),
        "device": str(t.device)
    })

pd.DataFrame(rows)


,이름,shape,dim,dtype,device
0,a 정수,"(3,)",1,torch.int64,cpu
1,b 실수,"(3,)",1,torch.float32,cpu
2,c 영행렬,"(2, 3)",2,torch.float32,cpu
3,d 난수,"(2, 3, 4)",3,torch.float32,cpu
4,e 스칼라,(),0,torch.float32,cpu


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [6]:
# 답안
# (3, 4) Tensor 생성
A = torch.tensor([[1., 2., 3., 4.],
                  [5., 6., 7., 8.],
                  [9., 10., 11., 12.]])

# (4, 2) Tensor 생성
B = torch.tensor([[1., 2.],
                  [3., 4.],
                  [5., 6.],
                  [7., 8.]])

# 행렬곱 결과 및 shape 출력
C = A @ B
print("행렬곱 결과:\n", C)
print("행렬곱 shape:", C.shape)

# 첫 번째 Tensor의 열별 평균 출력
print("열별 평균:", A.mean(dim=0))

# 첫 번째 Tensor의 행별 합계 출력
print("행별 합계:", A.sum(dim=1))


행렬곱 결과:
 tensor([[ 50.,  60.],
        [114., 140.],
        [178., 220.]])
행렬곱 shape: torch.Size([3, 2])
열별 평균: tensor([5., 6., 7., 8.])
행별 합계: tensor([10., 26., 42.])


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [8]:
# 답안
import time

# CPU와 GPU의 행렬곱 연산 시간 측정
def bench(device, n=2000, repeat=3):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)

    if device == "cuda":
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(repeat):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()  # GPU 연산 완료 대기
    return (time.time() - start) / repeat

# CPU 연산 시간 출력
print(f"CPU : {bench('cpu'):.4f} 초")

# GPU 연산 시간 출력
if torch.cuda.is_available():
    print(f"GPU : {bench('cuda'):.4f} 초")

CPU : 0.1417 초
GPU : 0.0059 초


###차이 설명
동일한 행렬곱 연산에서 CPU는 0.1417초, GPU는 0.0059초가 소요되어 GPU가 약 24배 빠르게 연산되었습니다. 이는 GPU가 대규모 행렬 연산을 병렬로 처리하는 데 유리하기 때문입니다.

## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [9]:
# 답안
# 1. y = 2x^3 + 5x, x = 3
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x**3 + 5 * x
y.backward()

# 자동미분과 손 계산 결과 비교
print("자동미분:", x.grad.item())
print("손 계산:", 6 * 3**2 + 5)


# 2. 직접 만든 수식: y = x^2 + 3x, x = 2
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()

# 자동미분과 손 계산 결과 비교
print("자동미분:", x.grad.item())
print("손 계산:", 2 * 2 + 3)


자동미분: 59.0
손 계산: 59
자동미분: 7.0
손 계산: 7
